# Evaluation: BiEncoder vs BM25


## Metrics

| Metric | What it measures |
|---|---|
| **MRR@10** | Mean Reciprocal Rank — average of 1/rank of the correct result |
| **R@1** | Was the top-1 result correct? (= precision / accuracy) |
| **R@5** | Was the correct result anywhere in the top 5? |
| **R@10** | Was the correct result anywhere in the top 10? |

## Expected results
- BM25 is a strong baseline for technical documents.
- BiEncoder should outperform BM25 on **semantic / paraphrase** queries where the exact keywords don't appear in the relevant chunk.

## Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/NLP PROJECT/Neural_Search_Engine-main'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    !pip install -q rank-bm25
else:
    PROJECT_ROOT = os.path.abspath('..')
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.tokenizer import BPETokenizer
from src.model import BiEncoder, EncoderConfig
from src.vector_store import VectorStore
from src.evaluate import evaluate_biencoder, evaluate_bm25, comparison_table, qualitative_comparison

print('Setup complete.')

Load Data and Models

In [2]:
# Load corpus
with open('data/processed/jurafsky_chunks_v2.json', encoding='utf-8') as f:
    chunks = json.load(f)

# Load evaluation set (10 hand-crafted queries)
eval_df = pd.read_csv('data/evaluation_set.csv')

print(f'Corpus chunks : {len(chunks)}')
print(f'Test queries  : {len(eval_df)}')
print()
print('Test queries:')
for i, row in eval_df.iterrows():
    print(f"  [{i+1}] {row['query'][:90]}")
    print(f"       expected: {row['expected_chunk_id']}")

Corpus chunks : 1410
Test queries  : 25

Test queries:
  [1] What is the standard notation for the activation function and hidden state of intermediate
       expected: chunk_0317
  [2] How can we compute the probability of a full sentence using bigram probabilities?
       expected: chunk_0062
  [3] How does an encoder-decoder model compute the probability of a target text given a source 
       expected: chunk_0783
  [4] Why is beam search preferred over greedy search and exhaustive search in machine translati
       expected: chunk_0695
  [5] How does BERT use the [CLS] token to predict the similarity score between a query and a do
       expected: chunk_0638
  [6] What is the concept of temperature sampling and how does it affect word probabilities?
       expected: chunk_0397
  [7] Who proposed the skip-gram and CBOW algorithms for neural net language models?
       expected: chunk_0294
  [8] Why is the raw dot product considered problematic as a similarity metric for very long do

In [ ]:
# Trained BPE tokenizer (saved by notebook 02).
tokenizer = BPETokenizer.load('checkpoints/tokenizer.json')

# 1) UNTRAINED encoder (random init) — same architecture, NO contrastive training.
#    This is the honest neural baseline: it shows how much the training actually adds.
model_untrained = BiEncoder(EncoderConfig(vocab_size=tokenizer.vocab_size))
model_untrained.eval()
store_untrained = VectorStore(model_untrained, tokenizer, batch_size=64)
store_untrained.build(chunks)

# 2) TRAINED encoder — the checkpoint from notebook 02 (config travels with the weights).
model_trained = BiEncoder.load('checkpoints/best.pt')
store_trained = VectorStore(model_trained, tokenizer, batch_size=64)
store_trained.build(chunks)

Compute Metrics for All Three Systems

In [ ]:
print('Evaluating BM25...')
bm25_metrics = evaluate_bm25(chunks, eval_df)

print('Evaluating UNTRAINED encoder (random init)...')
untrained_metrics = evaluate_biencoder(store_untrained, eval_df)

print('Evaluating TRAINED encoder...')
trained_metrics = evaluate_biencoder(store_trained, eval_df)

print('Done.')

In [ ]:
# Three-way comparison: keyword baseline vs untrained vs trained neural encoder.
metrics_keys = list(bm25_metrics.keys())
table = pd.DataFrame({
    'Metric':             metrics_keys,
    'BM25':               [bm25_metrics[k]      for k in metrics_keys],
    'Neural (untrained)': [untrained_metrics[k] for k in metrics_keys],
    'Neural (trained)':   [trained_metrics[k]   for k in metrics_keys],
})
table['Δ (trained − BM25)'] = (table['Neural (trained)'] - table['BM25']).round(4)

print('Results')
print(table.to_string(index=False))

Visualise Results

In [ ]:
x = np.arange(len(metrics_keys))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, table['BM25'],               width, label='BM25',               color='#aec7e8')
ax.bar(x,         table['Neural (untrained)'],  width, label='Neural (untrained)', color='#ffbb78')
ax.bar(x + width, table['Neural (trained)'],    width, label='Neural (trained)',   color='#98df8a')

ax.set_xticks(x)
ax.set_xticklabels(metrics_keys)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score (higher is better)')
ax.set_title('Retrieval evaluation: BM25 vs from-scratch neural encoder')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/evaluation_results.png', dpi=150)
plt.show()

---
## 4. Per-Query Analysis

In [ ]:
from src.evaluate import _tokenize_bm25, reciprocal_rank
from rank_bm25 import BM25Okapi

# Build BM25 once (not inside the loop)
corpus = [c['content'] for c in chunks]
chunk_ids = [c['id'] for c in chunks]
bm25 = BM25Okapi([_tokenize_bm25(d) for d in corpus])

header = '{:<60} {:>9} {:>11}  {}'.format('Query', 'BM25 rank', 'Neural rank', 'Winner')
print(header)
print('-' * 96)

for _, row in eval_df.iterrows():
    query = row['query']
    correct_id = row['expected_chunk_id']

    # BM25 rank of the correct chunk
    scores = bm25.get_scores(_tokenize_bm25(query))
    bm25_ranked = [chunk_ids[i] for i in sorted(range(len(scores)),
                                                key=lambda i: scores[i],
                                                reverse=True)[:10]]
    bm25_rank = next((i + 1 for i, cid in enumerate(bm25_ranked) if cid == correct_id), 11)

    # Neural (trained) rank of the correct chunk
    bi_results = store_trained.search(query, top_k=10)
    bi_ranked  = [r['id'] for r in bi_results]
    bi_rank    = next((i + 1 for i, cid in enumerate(bi_ranked) if cid == correct_id), 11)

    if bi_rank < bm25_rank:
        winner = 'Neural wins'
    elif bm25_rank < bi_rank:
        winner = 'BM25 wins'
    else:
        winner = 'tie'

    bm25_display = '>10' if bm25_rank == 11 else str(bm25_rank)
    bi_display   = '>10' if bi_rank   == 11 else str(bi_rank)
    print('{:<60} {:>9} {:>11}  {}'.format(query[:58], bm25_display, bi_display, winner))

Qualitative Comparison

In [ ]:
queries_to_show = eval_df['query'].tolist()[:2]

for query in queries_to_show:
    print('=' * 80)
    print(f'QUERY: {query}')
    print()

    bi_results = store_trained.search(query, top_k=3)

    corpus = [c['content'] for c in chunks]
    chunk_ids = [c['id'] for c in chunks]
    bm25 = BM25Okapi([_tokenize_bm25(d) for d in corpus])
    bm25_scores = bm25.get_scores(_tokenize_bm25(query))
    top3 = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:3]

    print(f'{"BM25 results":<50} | {"Neural (trained) results"}')
    print('-' * 100)
    for rank in range(3):
        bm_id = chunk_ids[top3[rank]]
        bm_text = corpus[top3[rank]][:80]
        bi_id = bi_results[rank]['id']
        bi_text = bi_results[rank]['content'][:80]
        print(f'[{rank+1}] {bm_id}: {bm_text}...')
        print(f'    Neural [{rank+1}] {bi_id}: {bi_text}...')
        print()
    print()

The eval set has three flavors of queries:
- **`keyword`** — 10 queries that share heavy vocabulary with their target chunk.
  BM25 should dominate here because keyword overlap is its native strength.
- **`paraphrase`** — 10 queries that target the SAME chunks as the keyword
  queries but phrased with synonyms. This is where BiEncoder should shine.
- **`paraphrase_new`** — 5 queries on different topics across the book,
  also written with paraphrased vocabulary.

In [ ]:
from src.evaluate import metrics_by_query_type

# Compute BM25 and TRAINED-encoder metrics, sliced by query_type.
by_type_trained = metrics_by_query_type(chunks, store_trained, eval_df)

# Pivot into a wide table: rows = (query_type, metric), columns = system.
pivoted = (
    by_type_trained
    .pivot_table(index=['query_type', 'metric'], columns='system', values='score')
    .reset_index()
)
pivoted['Delta (BiEnc - BM25)'] = (pivoted['BiEncoder'] - pivoted['BM25']).round(4)

print(pivoted.to_string(index=False))

In [ ]:
# Visualize MRR@10 per query_type
mrr_only = by_type_trained[by_type_trained['metric'] == 'MRR@10']
pivot_mrr = mrr_only.pivot_table(index='query_type', columns='system', values='score')

order = ['keyword', 'paraphrase', 'paraphrase_new']
pivot_mrr = pivot_mrr.reindex([o for o in order if o in pivot_mrr.index])

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(pivot_mrr))
width = 0.35
ax.bar(x - width/2, pivot_mrr['BM25'],     width, label='BM25',     color='#aec7e8')
ax.bar(x + width/2, pivot_mrr['BiEncoder'], width, label='Neural (trained)', color='#98df8a')

ax.set_xticks(x)
ax.set_xticklabels(pivot_mrr.index, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('MRR@10')
ax.set_title('Where BM25 wins vs where the neural encoder wins')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for i, qt in enumerate(pivot_mrr.index):
    ax.text(i - width/2, pivot_mrr.loc[qt, 'BM25'] + 0.02,
            f"{pivot_mrr.loc[qt, 'BM25']:.2f}", ha='center', fontsize=10)
    ax.text(i + width/2, pivot_mrr.loc[qt, 'BiEncoder'] + 0.02,
            f"{pivot_mrr.loc[qt, 'BiEncoder']:.2f}", ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('checkpoints/mrr_by_query_type.png', dpi=150)
plt.show()